# Setup

In [1]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "src")))
import pandas as pd
from processor.main import Processor
from processor.table.representation.impl.df_table import DFTable
from processor.table.store.table_store_factory import ImplementedTableStore

In [2]:
# OpenAI model
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv('OPENAI_API_KEY')
# model = 'gpt-4o-mini-2024-07-18'

# Local Model
model = "src/processor/model/weight/qwen25-7b"

embed_path = "src/processor/model/weight/bge-base"
processor = Processor(
    model, embed_path, ImplementedTableStore.DUCKDB_TABLE_STORE, "processor_output"
)

In [3]:
# Define constants
DB_SCHEMA = "E2E_SCHEMA"
DB_SCHEMA_AFTER_UNION = DB_SCHEMA + "_AFTER_UNION"
QUESTION_1 = "Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?"

In [4]:
# Prepare metadata (table descriptions)
metadata = pd.read_csv("data_src/buysite/metadata.csv")
table_descriptions: dict[str, str] = dict()
for i, row in metadata.iterrows():
    table_descriptions[row["table"]] = row["value"]

## Can be skipped if alread indexed

In [ ]:
# # Index the tables, retrieved by Pneuma for QUESTION_1
# from tqdm import tqdm
# processor.ctx.table_store.create_db_schema(DB_SCHEMA)

# TABLE_PATH_PREFIX = "data_src/buysite/dataset"
# retrieved_tables_q1 = [
#     "JI_ASN", "JI_ASN_CARRIER", "JI_ASN_LINE", "JI_PURCHASE_ORDER_LINE", "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS"
# ]
# for table_name in tqdm(retrieved_tables_q1):
#     df = pd.read_csv(os.path.join(TABLE_PATH_PREFIX, f"{table_name}.csv")).head(100000)
#     processor.ctx.table_store.add_table(
#         DB_SCHEMA,
#         table_name,
#         DFTable(df),
#         False,
#         True,
#     )

  0%|          | 0/5 [00:00<?, ?it/s]

 60%|██████    | 3/5 [00:00<00:00,  6.80it/s]/tmp/ipykernel_3171136/1383557893.py:10: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(TABLE_PATH_PREFIX, f"{table_name}.csv")).head(100000)
100%|██████████| 5/5 [04:03<00:00, 48.75s/it]


# Step 1: Schema Enhancement & Target Schema Generation

In [5]:
QUESTION_1

'Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?'

In [11]:
schema_generator_system_prompt =  """You are an expert in data integration. Your task is to:
1. Determine the target schema: the set of necessary columns required to directly answer a given question, without performing separate operations (e.g., calculate averages).
- The first column must always be the ID (primary key).
- The target schema must be self-sufficient: all essential attributes must be included so that the question can be answered without needing joins, external lookups, or additional tables.
- If the question involves counting items, quantities, or totals, ensure that the schema includes any necessary numeric fields (e.g., quantity per shipment) to correctly compute the answer.
- Be careful to exactly match any entity names (e.g., 'Amazon Same Day Delivery' versus 'Amazon') as stated in the question.

2. Simulate the SQL query that would answer the question using only the columns in the target schema.
- Assume the table is named "target_table"
- Your SQL must be valid and executable without any missing columns, as well as general enough (do not use features specific to certain implementations such as MySQL).

Output format:
{
  "schema": {
    "Column Name 1": {
      "description": "Description of column 1",
      "type": "DataType (e.g., INTEGER, VARCHAR, FLOAT)"
    },
    ...
  },
  "sql_query": "SQL query using only the schema above"
}

Important:
- Do not include any explanations or extra text outside the dictionary.
- Your output must be directly parseable as a Python dictionary."""

In [ ]:
from processor.conductor_state import ConductorState
from processor.utils.string_processor import parse_code_string

def get_target_schema(
        ctx: ConductorState,
        question: str,
        input_computation_nodes= [],
    ):
        ctx.logger.info(f"Getting target schema for the question {question}")
        messages = [
            {
                "role": "system",
                "content": schema_generator_system_prompt,
            },
            {"role": "user", "content": f"Question: {question}"},
        ]
        target_schema = ctx.llm.chat(messages=messages)
        ctx.logger.info(f"=> Target schema: {target_schema}")
        try:
            target_schema = parse_code_string(target_schema)
            return ctx.computation_graph.create_node(
                computation_description="Produced target schema for the given question.",
                computation_output=target_schema,
                input_nodes=input_computation_nodes,
            )
        except ValueError:
            ctx.logger.error(
                "Error encountered during target schema parsing, returning `{}`"
            )
            return dict()

In [13]:
target_schema_node = get_target_schema(processor.ctx, QUESTION_1)
target_schema = target_schema_node.computation_output
print(target_schema)

[2025-04-28 02:28:31] INFO in 3920900589: Getting target schema for the question Assuming all UPS ground shipments are delayed by 3 days before they are shipped (i.e., ship 3 days later than scheduled), how many items will be impacted?
[2025-04-28 02:29:13] INFO in 3920900589: => Target schema: {
  "schema": {
    "ID": {
      "description": "Unique identifier for each shipment",
      "type": "INTEGER"
    },
    "ScheduledShipDate": {
      "description": "The date the shipment was scheduled to be shipped",
      "type": "DATE"
    },
    "ActualShipDate": {
      "description": "The actual date the shipment was shipped",
      "type": "DATE"
    },
    "Carrier": {
      "description": "The carrier of the shipment",
      "type": "VARCHAR"
    },
    "Quantity": {
      "description": "The number of items in the shipment",
      "type": "INTEGER"
    }
  },
  "sql_query": "SELECT SUM(Quantity) AS ImpactedItems FROM target_table WHERE Carrier = 'UPS' AND ActualShipDate > DATE_ADD(Sc

In [9]:
table_descriptions_node = processor.get_table_descriptions(DB_SCHEMA, existing_descriptions=table_descriptions)
table_descriptions = table_descriptions_node.computation_output
print(table_descriptions)

Describing tables:   0%|          | 0/5 [00:00<?, ?it/s]

[2025-04-22 18:14:38] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-22 18:14:43] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-22 18:14:44] INFO in schema_processor: => Overall description of table 'JI_ASN': This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.


Describing tables:  20%|██        | 1/5 [00:06<00:26,  6.65s/it]

[2025-04-22 18:14:44] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-22 18:14:54] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-22 18:14:54] INFO in schema_processor: => Overall description of table 'JI_ASN_CARRIER': This table contains detailed information regarding advanced shipping notices associated with various carriers, including unique identifiers for shipments and organizations, carrier details, and timestamps for updates. It facilitates tracking of logistics and shipment processes over time, highlighting the complexity of managing shipping operations.


Describing tables:  40%|████      | 2/5 [00:16<00:25,  8.67s/it]

...

[2025-04-22 18:15:04] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-22 18:15:13] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-22 18:15:14] INFO in schema_processor: => Overall description of table 'JI_PURCHASE_ORDER_LINE': This table contains detailed information about purchase order lines, including identifiers for the organization, purchase order, and line items, as well as attributes like quantities, unit prices, and timestamps for transaction stages. It tracks statuses related to invoicing and fulfillment, along with flags for conditions such as awarded or cancelled items. The dataset also supports analysis of supplier performance and purchasing patterns, serving as a vital resource for managing procurement operations and informing strategic decision-making.


Describing tables:  80%|████████  | 4/5 [00:36<00:09,  9.61s/it]

[2025-04-22 18:15:14] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-04-22 18:15:21] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-04-22 18:15:22] INFO in schema_processor: => Overall description of table 'JI_FULFILLMENT_CENTER_TERMS_CONDITIONS': This table encapsulates essential details pertinent to the procurement process, specifically the terms and conditions tied to purchase orders issued by the University of Chicago. It outlines the requirements for order acceptance, including shipping instructions and fulfillment guidelines, and delineates payment terms, such as discount structures and payment timelines. Additionally, it serves as a reference for purchasing contacts, facilitating better communication and adherence to regulations, thereby providing a comprehensive framework for managing procurement-related transactions effectively.


Describing tables: 100%|██████████| 5/5 [00:44<00:00,  8.91s/it]

{'JI_ASN': 'This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.', 'JI_ASN_CARRIER': 'This table contains detailed information regarding advanced shipping notices associated with various carriers, including unique identifiers for shipments and organizations, carrier details, and timestamps for updates. It facilitates tracking of logistics and shipment processes over time, highlighting the complexity of managing shipping operations.', 'JI_ASN_LINE': 'This table captures detailed information regarding advanced shipping notices at a granular line-item level, including ASN ID, shipment line ID, purchase order line ID, and the quantity of items ship

In [ ]:
# enhanced_schemas_node = processor.get_enhanced_schemas(
#     DB_SCHEMA, table_descriptions, 3, [table_descriptions_node]
# )
# enhanced_schemas = enhanced_schemas_node.computation_output
# print(enhanced_schemas)

# Step 2: Base Table Producer

In [5]:
target_schema = {
    "Shipment ID": "Unique identifier for each shipment",
    "Scheduled Ship Date": "Original scheduled date for the shipment",
    "Delayed Ship Date": "Actual ship date after the 3-day delay",
    "Item Count": "Number of items in the shipment",
}
table_descriptions = {
    "JI_ASN": "This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.",
    "JI_ASN_CARRIER": "This table contains detailed information regarding advanced shipping notices associated with various carriers, including unique identifiers for shipments and organizations, carrier details, and timestamps for updates. It facilitates tracking of logistics and shipment processes over time, highlighting the complexity of managing shipping operations.",
    "JI_ASN_LINE": "This table captures detailed information regarding advanced shipping notices at a granular line-item level, including ASN ID, shipment line ID, purchase order line ID, and the quantity of items shipped. It also records timestamps for record creation and updates, facilitating effective tracking and management of shipments. This data is essential for monitoring the shipping process and ensuring accurate fulfillment against purchase orders within the supply chain.",
    "JI_PURCHASE_ORDER_LINE": "This table contains detailed information about purchase order lines, including identifiers for the organization, purchase order, and line items, as well as attributes like quantities, unit prices, and timestamps for transaction stages. It tracks statuses related to invoicing and fulfillment, along with flags for conditions such as awarded or cancelled items. The dataset also supports analysis of supplier performance and purchasing patterns, serving as a vital resource for managing procurement operations and informing strategic decision-making.",
    "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS": "This table encapsulates essential details pertinent to the procurement process, specifically the terms and conditions tied to purchase orders issued by the University of Chicago. It outlines the requirements for order acceptance, including shipping instructions and fulfillment guidelines, and delineates payment terms, such as discount structures and payment timelines. Additionally, it serves as a reference for purchasing contacts, facilitating better communication and adherence to regulations, thereby providing a comprehensive framework for managing procurement-related transactions effectively.",
}

In [6]:
relevant_table_ids_node = processor.select_relevant_table_ids(
    DB_SCHEMA,
    target_schema,
    table_descriptions,
    # 3,
    # [target_schema_node, table_descriptions_node],
)
relevant_table_ids = relevant_table_ids_node.computation_output
print(relevant_table_ids)

[2025-04-22 18:28:28] INFO in base_table_producer: Checking the relevance of table `JI_ASN`
[2025-04-22 18:28:35] INFO in base_table_producer: => Table relevancy output: To determine if the provided table is relevant for constructing the target schema, we can analyze the columns in the table and see how they might relate to the requirements specified in the target schema.

The columns in the provided table are:
- ASN_ID
- ASN_SHIPMENT_NUMBER
- ORG_ID
- SHIPMENT_DATE
- DELIVERY_DATE
- SHIPMENT_NOTES
- ELT_TS

The target schema requires the following columns:
- Shipment ID
- Scheduled Ship Date
- Delayed Ship Date
- Item Count

Now let's analyze the connections:

1. **Shipment ID**: The `ASN_ID` could serve as a unique identifier for each shipment. This can directly fulfill the requirement for the "Shipment ID".

2. **Scheduled Ship Date**: The `SHIPMENT_DATE` can reasonably match the "Original scheduled date for the shipment". Thus, this column partially fulfills the requirements.

3. *

## UNION

In [7]:
relevant_table_ids = ['JI_ASN', 'JI_ASN_CARRIER', 'JI_ASN_LINE', 'JI_PURCHASE_ORDER_LINE']

In [ ]:
# processor.ctx.table_store.delete_table(DB_SCHEMA, "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS")

In [10]:
union_operations_node = processor.produce_union_operations(
    DB_SCHEMA,
    table_descriptions,
    5,
    # [target_schema_node, table_descriptions_node],
)
union_operations = union_operations_node.computation_output
print(union_operations)

[2025-04-22 18:33:07] INFO in base_table_producer: => available_tables_formatted: - JI_ASN (This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.):
```col: ASN_ID | ASN_SHIPMENT_NUMBER | ORG_ID | SHIPMENT_DATE | DELIVERY_DATE | SHIPMENT_NOTES | ELT_TS
sample row 1: 8133755 | CHIL71437 | 1962414 | 2021/02/26 05:00:00.000000000 | nan | nan | 2023/11/04 03:30:32.330000000
sample row 2: 18898116 | CHIL92144 | 1962414 | 2024/05/17 04:00:00.000000000 | nan | nan | 2024/05/18 00:15:30.626000000
sample row 3: 15092645 | CHIL86695 | 1962414 | 2023/06/29 04:00:00.000000000 | 2023/06/30 04:00:00.000000000 | nan | 2023/11/04 03:30:32.330000000
sample row 4:

In [11]:
union_operations = [
    {
        "Output Table ID": "Union_1",
        "Tables": ["JI_ASN", "JI_ASN_CARRIER", "JI_ASN_LINE"],
        "Unified Schema": [
            "ASN_ID",
            "Shipment_Number",
            "Org_ID",
            "Shipment_Date",
            "Delivery_Date",
            "Shipment_Notes",
            "Carrier",
            "Shipment_Control_ID",
            "Shipment_Line_ID",
            "PO_Line_ID",
            "Quantity_Shipped",
            "Comments",
            "ELT_TS",
        ],
        "Mappings": {
            "JI_ASN": {
                "ASN_ID": "ASN_ID",
                "ASN_SHIPMENT_NUMBER": "Shipment_Number",
                "ORG_ID": "Org_ID",
                "SHIPMENT_DATE": "Shipment_Date",
                "DELIVERY_DATE": "Delivery_Date",
                "SHIPMENT_NOTES": "Shipment_Notes",
                "ELT_TS": "ELT_TS",
            },
            "JI_ASN_CARRIER": {
                "ASN_ID": "ASN_ID",
                "ORG_ID": "Org_ID",
                "CARRIER": "Carrier",
                "SHIPMENT_CONTROL_ID": "Shipment_Control_ID",
                "ELT_TS": "ELT_TS",
            },
            "JI_ASN_LINE": {
                "ASN_ID": "ASN_ID",
                "SHIPMENT_LINE_ID": "Shipment_Line_ID",
                "PO_LINE_ID": "PO_Line_ID",
                "ORG_ID": "Org_ID",
                "QUANTITY_SHIPPED": "Quantity_Shipped",
                "COMMENTS": "Comments",
                "ELT_TS": "ELT_TS",
            },
        },
    }
]

In [12]:
unioned_tables_node = processor.run_union_operations(
    processor.ctx.table_store.get_all_tables_in_db_schema(DB_SCHEMA),
    union_operations,
    # [union_operations_node],
)
unioned_tables = unioned_tables_node.computation_output
print(unioned_tables)

{'JI_PURCHASE_ORDER_LINE': <processor.table.representation.impl.df_table.DFTable object at 0x7f6aed8d4800>, 'Union_1': <processor.table.representation.impl.df_table.DFTable object at 0x7f6aec6251c0>}


/zp_more/project_data/pneuma/processor/src/processor/table/representation/impl/df_table.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = concat(dfs, ignore_index=True)


In [13]:
import pandas as pd
import numpy as np

def clean_duplicate_ids(df: pd.DataFrame, id_col='ID') -> pd.DataFrame:
    df = df.copy()
    non_id_cols = [col for col in df.columns if col != id_col]
    
    grouped = df.groupby(id_col)
    cleaned_rows = []
    
    for id_val, group in grouped:
        if len(group) == 1:
            cleaned_rows.append(group.iloc[0].to_dict())
            continue
        
        misalign_count = 0
        total_checks = 0
        combined = {}
        
        for col in non_id_cols:
            non_null_vals = group[col].dropna().unique()
            if len(non_null_vals) > 1:
                misalign_count += 1
            if len(non_null_vals) >= 1:
                total_checks += 1
            combined[col] = non_null_vals[0] if len(non_null_vals) > 0 else np.nan
        
        if total_checks > 0 and misalign_count == total_checks:
            cleaned_rows.extend(group.to_dict(orient='records'))
            continue
        
        combined[id_col] = id_val
        cleaned_rows.append(combined)
    
    return pd.DataFrame(cleaned_rows)[df.columns].reset_index(drop=True)

In [15]:
len(unioned_tables['Union_1'].get_data())

3703

In [20]:
# NEW FEATURE: ENTITY RESOLUTION
union_1_data = unioned_tables["Union_1"].get_data()
cleaned_union_1_data = clean_duplicate_ids(union_1_data, union_1_data.columns[0])
unioned_tables["Union_1"].data = cleaned_union_1_data

In [22]:
# Save to DB
processor.ctx.table_store.create_db_schema(DB_SCHEMA_AFTER_UNION)
for table_id, table in unioned_tables.items():
    processor.ctx.table_store.add_table(DB_SCHEMA_AFTER_UNION, table_id, table, False, True)

In [23]:
union_1_description = processor.ctx.llm.chat([{
    "role": "user", "content": f"Can you join these table descriptions into one? They've just recently been unioned together, i.e., the resulting table combines the schemas (and hence information) together. In other words, you are asked to describe the combined table. The descriptions: - {table_descriptions["JI_ASN_CARRIER"]}\n- {table_descriptions["JI_ASN"]}\n- {table_descriptions["JI_ASN_LINE"]}"
}])
union_1_description

'The combined table provides comprehensive details regarding advanced shipping notices (ASNs) across various levels of granularity, merging information previously found in separate schemas. It includes unique identifiers for shipments, organizations, and carriers, as well as essential fields such as ASN ID, shipment number, organization ID, shipment line ID, and purchase order line ID. The table also records timestamps for both shipment and delivery, alongside timestamps for record creation and updates. Optional notes can be included to provide additional context. This amalgamated data not only aids in tracking logistics and managing shipping operations but also supports enhanced visibility within the supply chain by ensuring accurate fulfillment against purchase orders and facilitating the monitoring of the shipping process over time.'

## JOIN

In [24]:
table_descriptions = {
    "JI_ASN": "This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.",
    "JI_ASN_CARRIER": "This table contains detailed information regarding advanced shipping notices associated with various carriers, including unique identifiers for shipments and organizations, carrier details, and timestamps for updates. It facilitates tracking of logistics and shipment processes over time, highlighting the complexity of managing shipping operations.",
    "JI_ASN_LINE": "This table captures detailed information regarding advanced shipping notices at a granular line-item level, including ASN ID, shipment line ID, purchase order line ID, and the quantity of items shipped. It also records timestamps for record creation and updates, facilitating effective tracking and management of shipments. This data is essential for monitoring the shipping process and ensuring accurate fulfillment against purchase orders within the supply chain.",
    "JI_PURCHASE_ORDER_LINE": "This table contains detailed information about purchase order lines, including identifiers for the organization, purchase order, and line items, as well as attributes like quantities, unit prices, and timestamps for transaction stages. It tracks statuses related to invoicing and fulfillment, along with flags for conditions such as awarded or cancelled items. The dataset also supports analysis of supplier performance and purchasing patterns, serving as a vital resource for managing procurement operations and informing strategic decision-making.",
    "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS": "This table encapsulates essential details pertinent to the procurement process, specifically the terms and conditions tied to purchase orders issued by the University of Chicago. It outlines the requirements for order acceptance, including shipping instructions and fulfillment guidelines, and delineates payment terms, such as discount structures and payment timelines. Additionally, it serves as a reference for purchasing contacts, facilitating better communication and adherence to regulations, thereby providing a comprehensive framework for managing procurement-related transactions effectively.",
    "Union_1": 'The combined table provides comprehensive details regarding advanced shipping notices (ASNs) across various levels of granularity, merging information previously found in separate schemas. It includes unique identifiers for shipments, organizations, and carriers, as well as essential fields such as ASN ID, shipment number, organization ID, shipment line ID, and purchase order line ID. The table also records timestamps for both shipment and delivery, alongside timestamps for record creation and updates. Optional notes can be included to provide additional context. This amalgamated data not only aids in tracking logistics and managing shipping operations but also supports enhanced visibility within the supply chain by ensuring accurate fulfillment against purchase orders and facilitating the monitoring of the shipping process over time.'
}

### 1. Produce Join Operations

In [26]:
def __format_available_tables(
        ctx,
        db_schema: str,
        num_rows: int,
        table_descriptions: dict[str, str],
    ):
        available_tables_formatted = ""
        table_mappings = ctx.table_store.get_all_tables_in_db_schema(db_schema)
        for table_id, table in table_mappings.items():
            table_description = table_descriptions[table_id]
            available_tables_formatted += f"""- {table_id} ({table_description}):
```{table.get_representation(num_rows, 42)}```\n"""

        available_tables_formatted = available_tables_formatted.strip()
        ctx.logger.info(f"=> available_tables_formatted: {available_tables_formatted}")
        return available_tables_formatted

In [32]:
base_table_producer_prompts = {
    "tables_selector": """You are an experienced data scientist. You are given:
- A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
- A target schema that needs to be constructed using one or more of the available tables.

Your task is to determine whether this table is **relevant** for constructing the target schema — either fully or partially. A table is considered relevant if it provides **any** useful information toward fulfilling the target schema, such as:
- Matching any of the target columns exactly,
- Providing a column that can be transformed into a target column,
- Contributing auxiliary information (e.g., geographic clues from `city` or `address` that help construct `Is in Bay Area`).

Err on the side of inclusion: if you think even **one** column might help, mark the table as **relevant**.

End your reasoning with the following exact format, to ease parsing:

Relevant: yes/no""",
    "row_extender_step_1": """You are an experienced data scientist. You are given:
- A list of tables, each with its schema, a short description, and a few sample rows.
- The pipe character (`|`) is used to separate both column names and values.

Your task is to **analyze and describe** what each table represents, and then identify **which tables describe the same kind of real-world entity or object** (such as people, products, companies, events, etc.).

Only group tables that:
- Refer to the same kind of entity
- Can be combined via **row extension** (i.e., vertical stacking)
- Even if the columns are not exactly the same, their rows should be logically stackable (e.g., two tables of products with different attributes)

Do **not** group tables that refer to different concepts/entities, even if they share similar-looking columns.

Finish with a list of compatible groups like:
Row extension groups: Group 1: Table_0, Table_2 Group 2: Table_3, Table_4 ... (or none if no combinations are found)""",
    "row_extender_step_2": """You are an experienced data scientist. You have already analyzed the tables and identified which ones can be unioned together because they refer to the same kind of real-world entity.

You are given:
- A list of tables (description + schemas + samples)
- Your own prior reasoning and a list of union groups (e.g., Group 1: Table_0, Table_2)

Your job is to create a JSON plan that shows how each group can be unioned.

Instructions:
- For each group, create a **unified schema** by merging **semantically equivalent** columns (e.g., "Customer_Rating" and "RATING" should both become "Rating")
- Use **simple, general, and meaningful** names for the unified columns (e.g., "Phone", "Address", "Rating", "Reviews")
- For each table, create a mapping from its original column names to the unified schema
- It's okay if some original columns do not exist in the unified schema — just leave them unmapped
- Do not include duplicate columns in the unified schema — each concept should appear only once

Output directly the following format without extra texts or explanations:

Format if row extension groups exist:
```json
[
  {
    "Output Table ID": "Union_1",
    "Tables": ["Table_0", "Table_2"],
    "Unified Schema": ["Column1", "Column2", ...],
    "Mappings": {
      "Table_0": {"OrigColA": "Column1", "OrigColB": "Column2", ...},
      "Table_2": {"ColX": "Column1", "ColY": "Column2", ...}
    }
  }
]```

Format if row extension groups are empty/none:
```json
[]```""",
    "join_planner": """You are a highly skilled data engineer. You are given:
- A list of tables (with descriptions, schemas, and sample rows)
- The goal is to **join all tables** together into a final unified table by **step-wise horizontal merging**.

Assumptions:
- All tables should be joinable via appropriate key columns, either directly or through intermediate tables.
- You can choose any join order as long as all tables are included by the end.
- You should identify the most appropriate **key columns** for joining each pair of tables based on semantics or value similarity.
- The operations will be carried out using either SQL or semantic joins.

Your task:
- Construct a step-by-step join plan as a **list of operations**, where each operation joins two tables (or previous join results).
- Each step should specify:
  - The two input tables, one of which may be a join result from the prior step.
  - The columns being used for the join
  - The resulting table name for that step (e.g., "Join_1", "Join_2", etc.)

Output your answer directly as a JSON object with the following format without any extra explanations or formatting:

```json
[
  {
    "Join Result": "Join_1",
    "Left Table": "Table_A",
    "Right Table": "Table_B",
    "Left Join Key": "Column_X",
    "Right Join Key": "Column_Y"
  },
  {
    "Join Result": "Join_2",
    "Left Table": "Join_1",
    "Right Table": "Table_C",
    "Left Join Key": "UserID",
    "Right Join Key": "Customer_ID"
  }
]```

Remember, no comments, extra explanations, or formatting.""",
    "classification_prompt": """You are a highly skilled data engineer. You are given:
- A description of a join operation between two tables.
- Sample values for each join key column from both tables.

Your task is to classify whether the join can be performed using a standard SQL join (e.g., matching IDs or exactly matching names), or if it requires a *semantic join*. A semantic join is needed when the values differ in representation — for example, abbreviations, name variations, different formats, or different languages — and require normalization, transformation, or external knowledge to align correctly.

Carefully examine the values. If they seem to not be exactly equal (not because of the fact that they are sample values), and some interpretation or resolution is needed to make the join work, it is a semantic join.

At the end of your reasoning, respond in the following format (for easy parsing):

- Operation classification: standard or semantic""",
    "std_join": """You are a highly skilled data engineer.
You are given two tables, represented by their IDs, descriptions, schemas, and sample rows.

Your goal is to create a SQL script (SQLite) to join these tables through a given left and right join keys. Refer to the IDs as identifiers in the script.

Output the SQLite script directly without any extra formatting or explanation.""",
}

In [28]:
from processor.utils.string_processor import parse_code_string


def produce_join_operations(
        ctx,
        db_schema: str,
        table_descriptions: dict[str, str],
        num_rows=3,
        input_computation_nodes: list = [],
    ):
        available_tables_formatted = __format_available_tables(
            ctx, db_schema, num_rows, table_descriptions
        )
        msg = [
            {"role": "system", "content": base_table_producer_prompts["join_planner"]},
            {"role": "user", "content": available_tables_formatted},
        ]
        join_operations: list[dict[str, str]] = parse_code_string(ctx.llm.chat(msg))
        ctx.logger.info(f"Join operations: {join_operations}")
        return ctx.computation_graph.create_node(
            computation_description="Produced join operations",
            computation_output=join_operations,
            input_nodes=input_computation_nodes,
        )

In [29]:
join_operations_node = produce_join_operations(
    processor.ctx,
    DB_SCHEMA_AFTER_UNION,
    table_descriptions,
    5,
    # [union_operations_node],
)
join_operations = join_operations_node.computation_output
print(join_operations)

[2025-04-22 18:56:48] INFO in 698260742: => available_tables_formatted: - JI_PURCHASE_ORDER_LINE (This table contains detailed information about purchase order lines, including identifiers for the organization, purchase order, and line items, as well as attributes like quantities, unit prices, and timestamps for transaction stages. It tracks statuses related to invoicing and fulfillment, along with flags for conditions such as awarded or cancelled items. The dataset also supports analysis of supplier performance and purchasing patterns, serving as a vital resource for managing procurement operations and informing strategic decision-making.):
```col: ORG_ID | PO_ID | PO_LINE_ID | DEPT_KEY | SUPPLIER_KEY | ITEM_KEY | PO_NUMBER | CONTRACT_ID | CONTRACT_NUMBER | QUANTITY | EXTENDED_PRICE | CREATED_TS | DISTRIBUTION_TS | EXPORT_TS | LASTREVISION_TS | ORIGINALREVISION_TS | WORKFLOWCOMPLETED_TS | ACCOUNTING_DATE | USER_OWNER_KEY | USER_SUBMITTER_KEY | EXTERNAL_PO_ID | LINE_NUMBER | UNIT_PRICE

### 2. Execute Join

In [30]:
join_operations = [
    {
        "Join Result": "Join_1",
        "Left Table": "JI_PURCHASE_ORDER_LINE",
        "Right Table": "Union_1",
        "Left Join Key": "PO_LINE_ID",
        "Right Join Key": "PO_Line_ID",
    }
]

In [35]:
from processor.utils.string_processor import parse_sql_string


def run_std_join_operation(
    ctx,
    left_table_id: str,
    right_table_id: str,
    left_table,
    right_table,
    left_join_key: str,
    right_join_key: str,
    input_nodes = [],
):
    """Runs a single standard join operation."""
    msg = [
        {
            "role": "system",
            "content": base_table_producer_prompts["std_join"],
        },
        {
            "role": "user",
            "content": f"""- Left table (ID: {left_table_id}; join key: {left_join_key}): {left_table.get_representation(3, 42)}
- Right table (ID: {right_table_id}; join key: {right_join_key}): {right_table.get_representation(3, 42)}""",
        },
    ]
    sql_script = parse_sql_string(ctx.llm.chat(msg))

    ctx.logger.info(f"=> Executing SQL: {sql_script}")
    joined_table = ctx.table_store.execute_sql_query(
        sql_query=sql_script,
        tables_involved={
            left_table_id: left_table,
            right_table_id: right_table,
        },
    )
    return ctx.computation_graph.create_node(
        f"Executing this SQL script for a standard join operation:\n{sql_script}",
        joined_table,
        input_nodes,
    )

In [36]:
def run_join_operations(
    ctx,
    table_mapping: dict,
    operations: list[dict[str, str]],
    num_values=3,
    input_computation_nodes = [],
):
    """
    Returns a list of operations to join tables (if any) within the DB schema.

    Args:
        ctx (ConductorState): Conductor state object.
        table_mappings (dict[str,AbstractTable]): The mapping between table IDs and table objects.
        operations (list[dict[str, Any]]): The join operations.
        num_values (int): Number of rows to sample for each table.
        input_computation_nodes (Node): A list of input nodes to keep track of computation.
    Returns:
        Output (Node[dict[str, AbstractTable]]): Computation node consisting of the table mappings after the operations have been applied.
    """

    # Ensure non-mutability of the original object
    table_mapping_copy = {k: v.copy() for k, v in table_mapping.items()}
    extra_input_nodes: list = []
    for operation in operations:
        join_table_id: str = operation["Join Result"]
        left_table_id: str = operation["Left Table"]
        right_table_id: str = operation["Right Table"]
        left_join_key: str = operation["Left Join Key"]
        right_join_key: str = operation["Right Join Key"]

        left_key_samples = table_mapping_copy[left_table_id].get_attribute_values(
            attr_name=left_join_key,
            num_values=num_values,
            random_seed=42,
        )
        right_key_samples = table_mapping_copy[right_table_id].get_attribute_values(
            attr_name=right_join_key,
            num_values=num_values,
            random_seed=42,
        )

        msg = [
            {
                "role": "system",
                "content": base_table_producer_prompts["classification_prompt"],
            },
            {
                "role": "user",
                "content": f"""- Samples of left join key ({left_join_key}): {left_key_samples}
- Samples of right join key ({right_join_key}): {right_key_samples}""",
            },
        ]
        classification_result = ctx.llm.chat(msg)
        ctx.logger.info(f"=> classification_result: {classification_result}")
        join_node = run_std_join_operation(
            ctx,
            left_table_id,
            right_table_id,
            table_mapping_copy[left_table_id],
            table_mapping_copy[right_table_id],
            left_join_key,
            right_join_key,
            input_computation_nodes,
        )
        extra_input_nodes.append(join_node)
        joined_table = join_node.computation_output
        table_mapping_copy[join_table_id] = joined_table
        del table_mapping_copy[left_table_id]
        del table_mapping_copy[right_table_id]
    return ctx.computation_graph.create_node(
        "Ran join operations over the tabless",
        table_mapping_copy,
        input_computation_nodes + extra_input_nodes,
    )

In [37]:
join_results_node = run_join_operations(
    processor.ctx,
    processor.ctx.table_store.get_all_tables_in_db_schema(DB_SCHEMA_AFTER_UNION),
    join_operations,
    5,
    # [join_operations_node],
)
join_results = join_results_node.computation_output
print(join_results)

[2025-04-22 19:02:40] INFO in 2211558722: => classification_result: In this case, the join keys from both tables, PO_LINE_ID from the left and PO_Line_ID from the right, have completely different sample values. There is no overlap in the provided samples, which suggests that they do not match at all.

Given that the values seem to represent different identifiers with no direct correlation or variation in naming, it indicates that either there might be separate systems at play or that the keys represent different entities. 

Since the samples of the join keys do not seem to be directly relatable without further transformation or mapping, and normalization would likely be necessary to align them correctly, it highlights that external knowledge about the relationship between these identifiers may be needed to successfully perform a join.

Therefore, this join is more complex than a standard SQL join and falls under the category of a semantic join.

- Operation classification: semantic
[20

In [39]:
processor.ctx.table_store.add_table(
    DB_SCHEMA_AFTER_UNION,
    "base_table",
    join_results['Join_1'],
    True,
    True,
)

# Base Table Reducer

In [ ]:
# processor.ctx.llm.chat([{
#     "role": "user", "content": """I have just merged two tables, and I have both their descriptions. It is a left join"""
# }])

In [40]:
target_schema

{'Shipment ID': 'Unique identifier for each shipment',
 'Scheduled Ship Date': 'Original scheduled date for the shipment',
 'Delayed Ship Date': 'Actual ship date after the 3-day delay',
 'Item Count': 'Number of items in the shipment'}

In [69]:
base_table_reducer_prompts = {
    "python_column_extractor": """You are an experienced data scientist. Given a table represented as a Pandas DataFrame, your task is to write a Python function named generate_column with no arguments except for the dataframe itself that returns a list representing the values of the new column based on this table. You are also given a question user has, which will help you determine what kind of computations that need to be done (e.g., adding time delta to the row values). Each element of the list should correspond to a row in the DataFrame.

Constraints:
- Only use columns that are present in the input DataFrame.
- To be safe, you should convert data types of the columns in your code before performing computation.
- Handle missing (null) values carefully and appropriately.
- Use regex to determine equality instead of "==".

Output the function directly without any extra explanations or formattings.""",
    "column_projection": """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema that we will transform the schema of source table into it in a step-by-step manner.
- A question that we want to answer, which was used to form the target schema.
- A column from the target schema as the current target column.

Your goal is to determine whether to select a certain column from SRC or extract information from certain column(s) from SRC to form the target column (even as simple as adding time delta to each row).
Extract_column can relies on external tools such as Python code interpreter, SQL processor, or LLM.

When using extract_column, always include all columns from SRC that are required to perform the extraction, even if their role seems minor or indirect. These can include helper columns (e.g., timestamps for computing durations, ZIP codes for locations, etc.).

The output format for selecting a certain column:
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID."
    "columns_involved": ["Restaurant ID"],
}

While for extracting information from certain column(s):
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`."
    "columns_involved": ["City", "ZIP Code"],
}

NOTE: consider the question very carefully when determining how to get the current target column, as it may cue what the target schema means.

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary.""",
    "extract_mode": """You are a data scientist working with structured tables.

You will be given:
- A table (schema and sample rows).
- A new column to generate, which is needed to answer a question.

Your job is to decide:
1. Should the values of the new column be extracted row-by-row using language reasoning by LLM?
2. Or, can the values be generated using a single Python function that processes the other column(s)?

Output one of:
- 'rowwise_extraction'
- 'python_code'

Output a JSON object directly without any quotes, explanations, or formatting with the following format:
{
    "type": "python_code/rowwise_extraction"
    "explanation": "Explanation of the computations that need to be done to produce the column."
}

Carefully interpret what the new column expects based on the given question, and prioritize python_code, which utilizes Pandas DataFrame, unless LLM is strictly necessary.""",
    "extract_col": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A column to be added to this table whose values depend on the other columns in the table.

Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

Output your result strictly as a Python list representing the new column values for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list.""",
    "reduce_row": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A user's question.
- A table that combines multiple source of information to answer the question.

Your goal is to produce a SQL code (DuckDB) containing predicates to reduce the rows of target_table. In other words, you need to eliminate irrelevant rows.
However, you should do so carefully, especially when comparing equality (e.g., use "LOWER(name) LIKE LOWER('%john%')").

Extra note:
- DuckDB already understands date columns, so you do not need to wrap such columns with `DATE()`.

Output your result strictly as a SQL code (DuckDB) without any extra formatting, explanations, or text. The output must be directly parseable as a SQL code.""",
}

In [89]:
column_extraction_new_prompt = """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema, which was designed to help answer a specific question.
- A single current target column, which is a key-value pair from the target schema (e.g., "Delayed Ship Date": "Actual ship date after the 3-day delay").

Your task is to decide whether to:
1. Select a column from SRC directly (if it maps cleanly to the target column), or
2. Extract or compute the target column using one or more columns from SRC (e.g., adding a time delta, parsing a field, applying a condition).

When deciding this, you must:
- Carefully analyze both the name and explanation of the current target column.
- Consider how the current column helps answer the question.
- Determine what transformation or logic is required to form this column from the source data.

When using extract_column, always include all columns from SRC that are required to perform the extraction, including:
- Main columns used in transformation.
- Helper/reference columns (e.g., timestamps, ZIP codes).
- Any columns used for conditional logic or filters (e.g., shipment method, status).

Be very careful about column names of SRC; do not exclude underscore symbols or whitespaces in the column names of SRC, as these result in errors.

Output format:

If you can directly select a column:
```python
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID.",
    "columns_involved": ["Restaurant ID"],
}
```

If the target column must be derived:
```python
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`.",
    "columns_involved": ["City", "ZIP Code"],
}
```

Your output must be a single valid Python dictionary. Do not add any extra text, markdown, or formatting."""

### First

In [90]:
from processor.conductor_state import ConductorState
from processor.table.representation.abstract_table import AbstractTable
from processor.utils.string_processor import clean_code_string, parse_code_string

import pandas as pd
import re


def compute_target_table(
        ctx: ConductorState,
        question: str,
        base_table: AbstractTable,
        target_schema: dict[str, str],
        num_rows=3,
        input_nodes = [],
    ):
        """
        Projects `base_table`, specifically its schema, to the `target_schema`,
        resulting in `target_table`.
        """
        ctx.logger.info("Computing target table")
        target_table_cols: dict[str, list] = dict()
        extra_input_nodes: list = []
        for col in target_schema:
            ctx.logger.info(f"=> Processing column {col}")
            msg = [
                {
                    "role": "system",
                    "content": column_extraction_new_prompt,
                },
                {
                    "role": "user",
                    "content": f"Source table: ```{base_table.get_representation(num_rows, 42)}```\nTarget Schema: ```{target_schema}```\n- Question: ```{question}```\n- Target Column: ```{col}: {target_schema[col]}```",
                },
            ]
            operation: dict[str, str] = parse_code_string(ctx.llm.chat(msg))
            ctx.logger.info(f"==> Operation: {operation}")
            if operation["operation"] == "select_column":
                operation_node = ctx.computation_graph.create_node(
                    "Mapped a column directly.",
                    list(base_table[operation["columns_involved"][0]]),
                    input_nodes,
                )
            else:
                ctx.logger.info("WARNING: ENTERING EXTRACT_COLUMN")
                operation_node = extract_column(
                    ctx,
                    question,
                    base_table,
                    operation["columns_involved"],
                    f"{col}: {target_schema[col]}",
                    10,
                    num_rows,
                    input_nodes,
                )
            extra_input_nodes.append(operation_node)
            target_table_cols[col] = operation_node.computation_output
        target_table = type(base_table).merge_columns(target_table_cols)
        return ctx.computation_graph.create_node(
            "Projected columns from base table to target table.",
            target_table,
            input_nodes + extra_input_nodes,
        )


def extract_column(
    ctx: ConductorState,
    question: str,
    base_table: AbstractTable,
    columns_involved: list[str],
    target_column: str,
    row_batch=10,
    num_rows = 3,
    input_nodes: list = [],
):
    ctx.logger.info(f"===> Operation extract_column")
    sql_script = "SELECT "
    for col in columns_involved:
        sql_script += f'"{col}", '
    sql_script = sql_script[:-2] + " FROM base_table;"

    ctx.logger.info(f"===> sql_script: {sql_script}")

    columns_involved_table = ctx.table_store.execute_sql_query(
        sql_script,
        {
            "base_table": base_table,
        },
    )
    unique_columns_involved_table = columns_involved_table.drop_duplicates()


    # Ask LLM whether to use row-wise extraction or Python code
    msg = [
        {
            "role": "system",
            "content": base_table_reducer_prompts["extract_mode"],
        },
        {
            "role": "user",
            "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Overall Schema: ```{list(base_table.get_schema())}```\n- New Column: ```{target_column}```\n-Question: ```{question}```",
        },
    ]
    output = ctx.llm.chat(msg).strip()
    ctx.logger.info(f"OUTPUT: {output}")
    extraction_mode: dict[str,str] = parse_code_string(output)
    ctx.logger.info(f"===> extraction_mode: {extraction_mode}")
    actual_values: list[str] = []
    if extraction_mode['type'] == "python_code":
        # Generate code from the LLM
        code_gen_msg = [
            {
                "role": "system",
                "content": base_table_reducer_prompts["python_column_extractor"],
            },
            {
                "role": "user",
                "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Target column: ```{target_column}```\n- User's question:\n```{question}```\n- Computation to do: ```{extraction_mode['explanation']}```",
            },
        ]
        code_str = ctx.llm.chat(code_gen_msg)
        ctx.logger.info(f"Python code to extract: {code_str}")
        code_str = clean_code_string(code_str)
        exec_globals = {}
        exec(code_str, exec_globals)

        # ctx.logger.info(f"exec_globals: {exec_globals}")
        import re
        import pandas as pd
        generated_func = exec_globals.get("generate_column")

        if not generated_func:
            raise ValueError("LLM did not return a valid 'generate_column' function.")

        actual_values = generated_func(columns_involved_table.get_data())

    return ctx.computation_graph.create_node(
        "Extraced column values from existing columns in the base table.",
        actual_values,
        input_nodes,
    )

In [91]:
table_descriptions = {
    "JI_ASN": "This table captures detailed information regarding advanced shipping notices (ASNs) at the document level, including essential fields such as ASN ID, shipment number, organization ID, and timestamps for both shipment and delivery, along with optional notes for additional context. The data supports effective tracking and management of shipping logistics, facilitating better supply chain visibility.",
    "JI_ASN_CARRIER": "This table contains detailed information regarding advanced shipping notices associated with various carriers, including unique identifiers for shipments and organizations, carrier details, and timestamps for updates. It facilitates tracking of logistics and shipment processes over time, highlighting the complexity of managing shipping operations.",
    "JI_ASN_LINE": "This table captures detailed information regarding advanced shipping notices at a granular line-item level, including ASN ID, shipment line ID, purchase order line ID, and the quantity of items shipped. It also records timestamps for record creation and updates, facilitating effective tracking and management of shipments. This data is essential for monitoring the shipping process and ensuring accurate fulfillment against purchase orders within the supply chain.",
    "JI_PURCHASE_ORDER_LINE": "This table contains detailed information about purchase order lines, including identifiers for the organization, purchase order, and line items, as well as attributes like quantities, unit prices, and timestamps for transaction stages. It tracks statuses related to invoicing and fulfillment, along with flags for conditions such as awarded or cancelled items. The dataset also supports analysis of supplier performance and purchasing patterns, serving as a vital resource for managing procurement operations and informing strategic decision-making.",
    "JI_FULFILLMENT_CENTER_TERMS_CONDITIONS": "This table encapsulates essential details pertinent to the procurement process, specifically the terms and conditions tied to purchase orders issued by the University of Chicago. It outlines the requirements for order acceptance, including shipping instructions and fulfillment guidelines, and delineates payment terms, such as discount structures and payment timelines. Additionally, it serves as a reference for purchasing contacts, facilitating better communication and adherence to regulations, thereby providing a comprehensive framework for managing procurement-related transactions effectively.",
    # "Union_1": 'The combined table provides comprehensive details regarding advanced shipping notices (ASNs) across various levels of granularity, merging information previously found in separate schemas. It includes unique identifiers for shipments, organizations, and carriers, as well as essential fields such as ASN ID, shipment number, organization ID, shipment line ID, and purchase order line ID. The table also records timestamps for both shipment and delivery, alongside timestamps for record creation and updates. Optional notes can be included to provide additional context. This amalgamated data not only aids in tracking logistics and managing shipping operations but also supports enhanced visibility within the supply chain by ensuring accurate fulfillment against purchase orders and facilitating the monitoring of the shipping process over time.'
}
target_schema = {
    "Shipment ID": "Unique identifier for each shipment",
    "Scheduled Ship Date": "Original scheduled date for the shipment",
    "Delayed Ship Date": "Actual ship date after the 3-day delay",
    "Item Count": "Number of items in the shipment",
}

In [92]:
target_table_node = compute_target_table(
    processor.ctx,
    QUESTION_1,
    processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, "base_table"),
    target_schema,
    2,
    # [join_operations_node],
)
target_table = target_table_node.computation_output
print(target_table)

[2025-04-22 19:51:28] INFO in 2059700159: Computing target table
[2025-04-22 19:51:28] INFO in 2059700159: => Processing column Shipment ID
[2025-04-22 19:51:29] INFO in 2059700159: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.Shipment_Control_ID as the unique identifier for each shipment.', 'columns_involved': ['Shipment_Control_ID']}
[2025-04-22 19:51:29] INFO in 2059700159: => Processing column Scheduled Ship Date
[2025-04-22 19:51:31] INFO in 2059700159: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.REQUESTED_DELIVERY_DATE as the original scheduled date for the shipment.', 'columns_involved': ['REQUESTED_DELIVERY_DATE']}
[2025-04-22 19:51:31] INFO in 2059700159: => Processing column Delayed Ship Date
[2025-04-22 19:51:32] INFO in 2059700159: ==> Operation: {'operation': 'extract_column', 'description': 'Calculate the Delayed Ship Date by adding 3 days to the SRC.`DISTRIBUTION_TS` for UPS ground shipments.', 'columns_involved': 

In [99]:
target_table.get_schema()

['Shipment ID', 'Scheduled Ship Date', 'Delayed Ship Date', 'Item Count']

In [103]:
target_table.get_data()[["Item Count"]].value_counts()    

Item Count
0             100000
Name: count, dtype: int64

In [64]:
target_table[["Item Count"]].value_counts()

Item Count
0             100000
Name: count, dtype: int64

In [75]:
processor.ctx.table_store.add_table(DB_SCHEMA_AFTER_UNION, "target_table", target_table, True, True)

### Predicate

In [105]:
def execute_sql_query(
        sql_query: str, tables_involved: dict[str, AbstractTable] = None
    ) -> AbstractTable:
        """
        [EXPERIMENTAL] Executes SQL query

        Args:
            sql_query (str): SQL query to execute
            tables_involved (list[AbstractTable]): OPTIONAL - Specify tables to query over (used by, e.g., PyTableStore)
        """
        import duckdb

        # Create an in-memory DuckDB connection
        conn = duckdb.connect(database=":memory:")

        # Make sure you pass a dictionary of DFTable
        if tables_involved is None or len(tables_involved) == 0:
            raise ValueError(
                "PyTableStore requires `tables_involved` to execute SQL queries."
            )

        if not isinstance(tables_involved[list(tables_involved.keys())[0]], DFTable):
            raise ValueError("Only Pandas DataFrame is supported for now.")

        # Register each DataFrame as a DuckDB view
        for table_id, table in tables_involved.items():
            df = table.get_data().copy()
            for col in df.columns:
                if df[col].dtype == "object":
                    try:
                        df[col] = pd.to_datetime(df[col])
                    except Exception:
                        pass  # Not datetime, ignore
            conn.register(table_id.lower(), df)

        # Run your SQL query (assumed lowercase)
        data = conn.execute(sql_query).fetchdf()
        return DFTable(data)

In [106]:
from processor.utils.string_processor import parse_sql_string


def apply_predicate_to_target_table(
        ctx: ConductorState,
        target_table: AbstractTable,
        question: str,
        num_rows=3,
        input_nodes = [],
    ):
        """
        Applies a natural-language predicate to target table.
        """
        msg = [
            {"role": "system", "content": base_table_reducer_prompts["reduce_row"]},
            {
                "role": "user",
                "content": f"""- Table: ```{target_table.get_representation(num_rows, 42)}```
- Question: {question}\n- Table Schema (careful with column names, e.g., do not forget whitespaces if any): ```{target_table.get_schema()}```""",
            },
        ]
        llm_output = ctx.llm.chat(msg)
        sql_query = parse_sql_string(llm_output)
        ctx.logger.info(f"SQL Query: {sql_query}")
        final_table = execute_sql_query(
            sql_query, {"target_table": target_table}
        )
        return ctx.computation_graph.create_node(
            f"Applied this predicate to the rows of target table: {sql_query}",
            final_table,
            input_nodes,
        )

In [107]:
final_table_node = apply_predicate_to_target_table(
    processor.ctx,
    processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, "base_table"),
    QUESTION_1,
    5,
)
final_table = final_table_node.computation_output
print(final_table)

[2025-04-22 20:03:02] INFO in 1467923155: SQL Query: SELECT COUNT(*) 
FROM target_table 
WHERE LOWER(SHIPPING_METHOD) LIKE LOWER('%UPS%') 
AND DELIVERY_DATE > DATE(DISTRIBUTION_TS) + INTERVAL '3 days';


/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])
/tmp/ipykernel_2478959/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a 

BinderException: Binder Error: No function matches the given name and argument types 'lower(DOUBLE)'. You might need to add explicit type casts.
	Candidate functions:
	lower(VARCHAR) -> VARCHAR


LINE 3: WHERE LOWER(SHIPPING_METHOD) LIKE LOWER('%UPS%') 
              ^

In [104]:
target_schema

{'Shipment ID': 'Unique identifier for each shipment',
 'Scheduled Ship Date': 'Original scheduled date for the shipment',
 'Delayed Ship Date': 'Actual ship date after the 3-day delay',
 'Item Count': 'Number of items in the shipment'}

In [76]:
QUESTION_1

'Assuming that all shipments by UPS ground service are late by 3 days, how many items will be impacted?'

In [ ]:
# AND DATE("Delivery Date") > "Ship Date" + INTERVAL 3 day

In [50]:
x = execute_sql_query(
    """SELECT *
FROM target_table 
WHERE LOWER("Courier Service") LIKE LOWER('%UPS ground%')
AND "Delivery Date" > "Ship Date" + INTERVAL 3 day;""",
    {
        "target_table": processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, "target_table")
    }
)

/tmp/ipykernel_2195825/4078682329.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col])


In [51]:
x.data

,Shipment ID,Courier Service,Delivery Date,Ship Date


# Visualization

In [ ]:
from pyvis.network import Network
import networkx as nx

from processor.computation_graph import ComputationGraph


def interactive_network_pyvis(graph: ComputationGraph):
    G = nx.DiGraph()

    for node in graph.nodes:
        # label = f"{node.function_name}()\n{node.class_name or ''}\n{node.computation_description}"
        label = f"{node.function_name}()"
        G.add_node(node.id, label=label)

    for node in graph.nodes:
        for input_node in node.input_nodes:
            G.add_edge(input_node.id, node.id)

    net = Network(
        notebook=True,
        height="600px",
        width="100%",
        directed=True,
        cdn_resources="in_line",
    )
    net.from_nx(G)
    net.show("graph.html")  # Will now render inline in Jupyter

In [ ]:
interactive_network_pyvis(processor.ctx.computation_graph)

In [ ]:
# BACKUP OLD
enhanced_schemas = {
    "JI_ASN_CARRIER": [
        "ShippingNoticeID",
        "OrganizationID",
        "SHIPPER_CARRIER",
        "Shipment_Domain",
        "SHIPMENT_UNIQUE_ID",
        "ASN_Timestamp",
    ],
    "JI_PURCHASE_ORDER_LINE": [
        "ORGANIZATION_ID",
        "PURCHASE_ORDER_ID",
        "PO_LINE_ITEM_ID",
        "DEPARTMENT_KEY",
        "SUPPLIER_ID",
        "LINE_ITEM_KEY",
        "PURCHASE_ORDER_NUMBER",
        "CONTRACT_REFERENCE_ID",
        "CONTRACT_ID_NUMBER",
        "ORDER_QUANTITY",
        "TOTAL_LINE_COST",
        "ORDER_CREATION_TIMESTAMP",
        "DISTRIBUTION_SHIPMENT_TS",
        "EXPORT_TIMESTAMP",
        "LAST_REVISION_TIMESTAMP",
        "ORIGINAL_REVISION_TIMESTAMP",
        "WORKFLOW_COMPLETION_TS",
        "ACCOUNTING_DATE_TIMESTAMP",
        "USER_OWNER_ID",
        "USER_SUBMITTER_ID",
        "EXTERNAL_PURCHASE_ORDER_LINE_ID",
        "PO_LINE_NUMBER",
        "UNIT_PRICE_PER_ITEM",
        "CONTRACT_UNIT_PRICE_CONTRACTED",
        "SUPPLIER_ACCOUNTING_CODE",
        "SHIPMENT_METHOD",
        "IS_PO_LINE_AWARDED_BID_FLAG",
        "IS_PO_LINE_REJECTED_FLAG",
        "IS_PO_LINE_CANCELLED_FLAG",
        "IS_LINE_SENT_TO_SUPPLIER",
        "HAS_INVOICES",
        "IS_FORCE_MATCHED",
        "IS_PO_LINE_FORCED_MATCHED",
        "REQUISITION_REQUEST_ID",
        "REQUISITION_IDENTIFIER",
        "REQUISITION_LINE_NUMBER",
        "REQUISITION_LINE_ID",
        "REQUISITION_CREATION_TS",
        "PO_LATEST_REVISION_NUMBER",
        "HAS_REJECTED_RECEIPTS",
        "HAS_CREDITS",
        "IS_OVERSHIPPED",
        "IS_PO_LINE_EXCESS_RECEIPT",
        "IS_PO_LINE_OVERINVOICED",
        "HasSubstitutedInvoiceItems",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_HAS_CANCELLED_ITEMS",
        "HAS_RECEIVED_SHIPMENTS",
        "HAS_RETURN_RECEIPTS",
        "HAS_INVOICES",
        "UNIT_PRICE_IN_USD",
        "EXTENDED_PRICE_IN_USD",
        "CONTRACT_UNIT_PRICE_IN_USD",
        "SUPPLIER_RANKING",
        "IS_DIVERSE_SUPPLIER",
        "LIST_PRICE_SET_KEY",
        "LIST_PRICE_SET_DESCRIPTION",
        "CONTRACT_LIST_PRICE\n\nThis_name_better_reflects_that_the_value_in_this_column_is_likely_the_list_price_associated_with_the_contract_for_the_item,_which_is_specific_to_the_context_of_the_purchase_order_and_its_line_items._It_also_maintains_clarity_and_consistency_with_other_column_names_that_describe_pricing_aspects.",
        "LIST_PRICE_SET_VERSION_NUMBER",
        "PREVIOUS_LIST_PRICE_VERSION",
        "PREVIOUS_LIST_PRICE_SET_VERSION_NAME",
        "Purchase_Order_Type_Code",
        "Purchase_Order_Type",
        "RECEIPT_STATUS_ENUM",
        "RECEIPT_STATUS",
        "PO_Invoice_Status_Type",
        "INVOICE_STATUS",
        "PO_Workflow_Status_Type",
        "Purchase_Order_Workflow_Status",
        "PO_Match_Status_Enum",
        "PURCHASE_ORDER_MATCH_STATUS",
        "UNIT_PRICE_SOURCE_TYPE",
        "UNIT_PRICE_SOURCE_DESCRIPTION",
        "PO_LINE_MATCH_STATUS_DESCRIPTION",
        "PO_LINE_MATCHING_STATUS",
        "Contract_Unit_Price_Business",
        "CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY_CODE",
        "CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE_DESCRIPTION",
        "EXTENDED_PRICE_BUSINESS_VALUE",
        "BUSINESS_EXTENDED_PRICE_CURRENCY",
        "EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE_VALUE",
        "BUSINESS_UNIT_PRICE",
        "BUSINESS_UNIT_PRICE_CURRENCY",
        "EXCHANGE_RATE_BUSINESS_TO_USD",
        "HAS_SHIPPED",
        "HAS_RECEIVED_SHIPMENTS",
        "PO_LINE_RECEIPT_STATUS_DESCRIPTION\n\nThis_name_provides_clarity_about_the_nature_of_the_data_stored_in_the_column,_indicating_that_it_describes_the_status_of_receipts_for_each_PO_line.",
        "RECEIPT_STATUS_ENUM",
        "PO_LINE_SHIPMENT_STATUS_DESCRIPTION",
        "SHIPMENT_STATUS",
        "MAX_UNIT_PRICE_RECEIVED",
        "MINIMUM_RECEIPT_UNIT_PRICE",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_REQUIRES_RECEIPT_MATCHING",
        "IS_PO_LINE_MATCHING_REQUIRES_RECEIPT",
        "IS_PO_LINE_SHIPPED_IN_EXCESS",
        "MAX_RECEIPT_UNIT_PRICE_USD",
        "MIN_RECEIPT_UNIT_PRICE_USD",
        "SHIP_TO_ADDRESS_IDENTIFIER",
        "Bill_To_Address_ID",
        "COMMODITY_CODE_KEY",
        "REQUESTED_DELIVERY_DATE",
        "DELIVERY_DATE_CATEGORY",
        "DELIVERY_LEAD_TIME_IN_DAYS",
        "FORM_DOCUMENT_ID",
        "FORM_REQUEST_ID",
        "TOTAL_PURCHASE_ORDER_AMOUNT",
        "TOTAL_DOCUMENT_AMOUNT",
        "Document_Currency_Type",
        "Exchange_Rate_Grand_Total_Document",
        "Total_USD",
        "LAST_TRANSFORM_TS",
        "FULFILLMENT_CENTER_ID",
        "TOP_LEVEL_CATEGORY_NAME",
        "UNSPSC_Category_Level_1",
        "CATEGORY_LEVEL_2_DESCRIPTION",
        "UNSPSC_Category_Level_2",
    ],
    "JI_ORDER_ACK_LINE": [
        "Order_Acknowledgment_ID",
        "ShipmentLineIdentifier",
        "PURCHASE_ORDER_LINE_ID",
        "OrganizationID",
        "ORDER_QUANTITY",
        "EstimatedShippingDate",
        "Order_Status_Code",
        "Order_Ack_Status",
        "ACKNOWLEDGEMENT_NOTES",
        "LAST_UPDATED_TS",
    ],
    "JI_ASN_LINE": [
        "ASN_Line_ID",
        "ASN_LINE_ID",
        "PURCHASE_ORDER_LINE_ID",
        "Organization_ID",
        "Shipped_Quantity",
        "SHIPMENT_NOTES",
        "SHIPMENT_RECORD_TIMESTAMP",
    ],
    "JI_ASN": [
        "AdvancedShippingNoticeID",
        "Shipment_ID",
        "Originating_Organization_ID",
        "ScheduledShipmentDate",
        "DELIVERY_DATE",
        "SHIPMENT_COMMENTS",
        "LAST_UPDATE_TS",
    ],
}